# 💰 05 — Simulação de Decisão de Negócio

> **Credit Risk ML** · Notebook 5 de 5

**Objetivo:** conectar o modelo ao mundo real — transformar probabilidades em decisões e medir o impacto financeiro.

| Etapa | Descrição |
|-------|-----------|
| 5.1 | Sistema de decisão (threshold) |
| 5.2 | Simulação financeira por modelo |
| 5.3 | Otimização de threshold por lucro |
| 5.4 | Análise do threshold ótimo |
| 5.5 | Exemplos de decisão individual |
| 5.6 | Resumo final do pipeline |

---

### Lógica de Decisão

```
  score_default > threshold  →  REJEITAR crédito
  score_default ≤ threshold  →  APROVAR  crédito
```

### Impacto Financeiro (configurável)

| Decisão | Cliente Real | Resultado |
|---------|-------------|----------|
| APROVAR | Bom (0) | **+R$ 100** |
| APROVAR | Ruim (1) | **−R$ 500** |
| REJEITAR | Bom (0) | −R$ 10 (oportunidade perdida) |
| REJEITAR | Ruim (1) | R$ 0 (prejuízo evitado) |

---

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib
from dataclasses import dataclass
from sklearn.metrics import confusion_matrix
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#161B22','axes.edgecolor':'#30363D',
    'axes.labelcolor':'#E6EDF3','xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#E6EDF3','grid.color':'#21262D','grid.linewidth':0.8,
    'figure.dpi':120,'font.family':'monospace',
})
C = {'lr':'#58A6FF','rf':'#3FB950','xgb':'#F78166',
     'accent':'#D2A8FF','warn':'#E3B341','muted':'#8B949E'}
SEED = 42
print('✅  Imports OK')

In [ ]:
# Carrega artefatos
splits      = joblib.load('../models/splits.pkl')
model_res   = joblib.load('../models/model_results.pkl')
best_model  = joblib.load('../models/best_model.pkl')
scaler      = joblib.load('../models/scaler.pkl')

X_test_sc   = splits['X_test_sc']
y_test      = splits['y_test']
features    = splits['features']
all_results = model_res['all_results']

print(f'✅  Melhor modelo: {model_res["best_model_name"]} (AUC={model_res["best_model_auc"]:.4f})')

---
### 5.1 Sistema de Decisão

In [ ]:
@dataclass
class BusinessConfig:
    """Configuração financeira do sistema de decisão."""
    threshold:        float = 0.70
    profit_good:      float = 100.0
    loss_bad:         float = 500.0
    opportunity_cost: float = 10.0
    currency:         str   = 'BRL'


def make_decision(score: float, cfg: BusinessConfig) -> str:
    """Converte score de default em decisão de crédito."""
    return 'REJEITAR' if score > cfg.threshold else 'APROVAR'


def simulate_business(model, X_test, y_test, cfg: BusinessConfig) -> dict:
    """Simula o impacto financeiro completo de um modelo."""
    y_prob    = model.predict_proba(X_test)[:, 1]
    y_true    = np.array(y_test)
    approved  = y_prob <= cfg.threshold

    tn = int(np.sum( approved & (y_true==0)))  # bom aprovado   ✅
    fn = int(np.sum( approved & (y_true==1)))  # ruim aprovado  ❌
    fp = int(np.sum(~approved & (y_true==0)))  # bom rejeitado  ⚠️
    tp = int(np.sum(~approved & (y_true==1)))  # ruim rejeitado 🛡️

    lucro = tn*cfg.profit_good - fn*cfg.loss_bad - fp*cfg.opportunity_cost
    n     = len(y_true)

    return {
        'total':         n,
        'aprovados':     int(approved.sum()),
        'rejeitados':    int((~approved).sum()),
        'bom_aprovado':  tn,
        'ruim_aprovado': fn,
        'bom_rejeitado': fp,
        'ruim_rejeitado':tp,
        'lucro_total':   lucro,
        'lucro_cliente': lucro / n,
        'taxa_aprovacao':float(approved.mean()),
        'y_prob':        y_prob,
    }

cfg = BusinessConfig(threshold=0.70)
print('✅  Sistema de decisão configurado')
print(f'   Threshold    : {cfg.threshold}')
print(f'   Lucro (bom)  : +{cfg.profit_good} {cfg.currency}')
print(f'   Prejuízo     : -{cfg.loss_bad} {cfg.currency}')
print(f'   Oport. perdida: -{cfg.opportunity_cost} {cfg.currency}')

---
### 5.2 Simulação Financeira por Modelo

In [ ]:
model_map = {r['name']: r['model'] for r in all_results}
sim_map   = {}

for name, model in model_map.items():
    sim_map[name] = simulate_business(model, X_test_sc, y_test, cfg)

for name, r in sim_map.items():
    sign = '+' if r['lucro_total'] >= 0 else ''
    print(f'\n  ── {name}')
    print(f'     Aprovados         : {r["aprovados"]:>7,}  ({r["taxa_aprovacao"]:.1%})')
    print(f'     ✅  Bom aprovado   : {r["bom_aprovado"]:>7,}  × +{cfg.profit_good:.0f}  = '
          f'+{r["bom_aprovado"]*cfg.profit_good:>12,.0f} {cfg.currency}')
    print(f'     ❌  Ruim aprovado  : {r["ruim_aprovado"]:>7,}  × -{cfg.loss_bad:.0f}  = '
          f'-{r["ruim_aprovado"]*cfg.loss_bad:>12,.0f} {cfg.currency}')
    print(f'     ⚠️   Bom rejeitado  : {r["bom_rejeitado"]:>7,}  × -{cfg.opportunity_cost:.0f}   = '
          f'-{r["bom_rejeitado"]*cfg.opportunity_cost:>12,.0f} {cfg.currency}')
    print(f'     💰  Lucro total    :          {sign}{r["lucro_total"]:>14,.0f} {cfg.currency}')
    print(f'     💰  Lucro/cliente  :          {sign}{r["lucro_cliente"]:>14.2f} {cfg.currency}')

In [ ]:
# Visualização financeira
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0F1117')

names  = list(sim_map.keys())
colors = [C['lr'],C['rf'],C['xgb']]

# Lucro total
ax1 = axes[0]
lucros = [sim_map[n]['lucro_total']/1000 for n in names]
bars   = ax1.bar(names, lucros, color=colors, edgecolor='#0F1117', width=0.5, alpha=0.9)
ax1.axhline(0, color='#8B949E', lw=1)
for bar, v in zip(bars, lucros):
    sign = '+' if v >= 0 else ''
    ax1.text(bar.get_x()+bar.get_width()/2,
             bar.get_height() + (max(lucros)*0.03 if v >= 0 else min(lucros)*0.03),
             f'{sign}{v:,.1f}k', ha='center', fontsize=11, fontweight='bold')
ax1.set_title(f'Lucro Total (threshold={cfg.threshold})',
              fontsize=12, fontweight='bold', color='#E6EDF3')
ax1.set_ylabel(f'×1000 {cfg.currency}'); ax1.grid(axis='y', alpha=0.2)
ax1.tick_params(axis='x', labelrotation=10, labelsize=9)

# Taxa de aprovação
ax2 = axes[1]
taxas = [sim_map[n]['taxa_aprovacao']*100 for n in names]
bars2 = ax2.bar(names, taxas, color=colors, edgecolor='#0F1117', width=0.5, alpha=0.9)
for bar, v in zip(bars2, taxas):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax2.set_title('Taxa de Aprovação',fontsize=12,fontweight='bold',color='#E6EDF3')
ax2.set_ylabel('%'); ax2.grid(axis='y', alpha=0.2)
ax2.tick_params(axis='x', labelrotation=10, labelsize=9)

# Decomposição lucro — melhor modelo
ax3    = axes[2]
best_n = max(sim_map, key=lambda n: sim_map[n]['lucro_total'])
r      = sim_map[best_n]
comps  = {
    f'✅ Bom ap.\n(+{cfg.profit_good:.0f})':  r['bom_aprovado']*cfg.profit_good/1000,
    f'❌ Ruim ap.\n(-{cfg.loss_bad:.0f})':   -r['ruim_aprovado']*cfg.loss_bad/1000,
    f'⚠️ Bom rej.\n(-{cfg.opportunity_cost:.0f})': -r['bom_rejeitado']*cfg.opportunity_cost/1000,
}
bar_colors2 = [C['rf'], C['xgb'], C['warn']]
ax3.bar(comps.keys(), comps.values(), color=bar_colors2, edgecolor='#0F1117', width=0.6, alpha=0.9)
ax3.axhline(0, color='#8B949E', lw=1)
ax3.set_title(f'Componentes do Lucro\n({best_n})',
              fontsize=11, fontweight='bold', color='#E6EDF3')
ax3.set_ylabel(f'×1000 {cfg.currency}'); ax3.grid(axis='y', alpha=0.2)

plt.suptitle('Simulação de Impacto Financeiro',
             fontsize=15, color=C['accent'], fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/business_simulation.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
### 5.3 Otimização de Threshold por Lucro

In [ ]:
def optimize_threshold(model, X_test, y_test, cfg_base, steps=200):
    """Encontra o threshold que maximiza o lucro total."""
    thresholds = np.linspace(0.01, 0.99, steps)
    profits    = []
    for t in thresholds:
        cfg_t = BusinessConfig(
            threshold=t, profit_good=cfg_base.profit_good,
            loss_bad=cfg_base.loss_bad, opportunity_cost=cfg_base.opportunity_cost
        )
        res = simulate_business(model, X_test, y_test, cfg_t)
        profits.append(res['lucro_total'])
    profits    = np.array(profits)
    best_idx   = np.argmax(profits)
    return thresholds[best_idx], thresholds, profits

# Otimiza para todos os modelos
opt_results = {}
for name, model in model_map.items():
    best_t, thresholds, profits = optimize_threshold(model, X_test_sc, y_test, cfg)
    opt_results[name] = {'best_t': best_t, 'thresholds': thresholds, 'profits': profits}
    best_profit = simulate_business(model, X_test_sc, y_test,
                                    BusinessConfig(threshold=best_t,
                                                   profit_good=cfg.profit_good,
                                                   loss_bad=cfg.loss_bad,
                                                   opportunity_cost=cfg.opportunity_cost))['lucro_total']
    default_profit = sim_map[name]['lucro_total']
    print(f'  {name:<22} threshold ótimo={best_t:.3f}  '
          f'lucro={best_profit:,.0f}  '
          f'(vs {default_profit:,.0f} com 0.70  Δ={best_profit-default_profit:+,.0f})')

In [ ]:
# Curvas de lucro × threshold para todos os modelos
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0F1117')

for name, color in zip(names, colors):
    r  = opt_results[name]
    ax.plot(r['thresholds'], r['profits']/1000, lw=2.2, color=color, label=name)
    ax.scatter([r['best_t']], [r['profits'].max()/1000],
               color=color, zorder=5, s=80, marker='*')

ax.axvline(0.70, color=C['warn'], lw=1.5, ls='--', label='Threshold padrão (0.70)')
ax.axhline(0, color='#8B949E', lw=1, ls=':')
ax.set_xlabel('Threshold de Decisão', fontsize=12)
ax.set_ylabel('Lucro Total (×1000 BRL)', fontsize=12)
ax.set_title('Curva Lucro × Threshold — Todos os Modelos',
             fontsize=14, fontweight='bold', color='#E6EDF3')
ax.legend(fontsize=10, framealpha=0.3); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../reports/profit_curves.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
### 5.4 Análise do Threshold Ótimo

In [ ]:
# Melhor modelo + threshold ótimo → análise detalhada
best_model_name = model_res['best_model_name']
best_opt = opt_results[best_model_name]
best_t   = best_opt['best_t']

cfg_opt = BusinessConfig(threshold=best_t, profit_good=cfg.profit_good,
                          loss_bad=cfg.loss_bad, opportunity_cost=cfg.opportunity_cost)
r_opt   = simulate_business(best_model, X_test_sc, y_test, cfg_opt)
r_def   = sim_map[best_model_name]

print(f'  ── {best_model_name} ──────────────────────────────────')
print(f'\n  {'Métrica':<25} {'Threshold=0.70':>15}  {'Ótimo ({:.3f})'.format(best_t):>15}')
print(f'  {"─"*58}')
print(f'  {"Taxa de aprovação":<25} {r_def["taxa_aprovacao"]:>14.1%}  {r_opt["taxa_aprovacao"]:>14.1%}')
print(f'  {"Bons aprovados":<25} {r_def["bom_aprovado"]:>15,}  {r_opt["bom_aprovado"]:>15,}')
print(f'  {"Ruins aprovados":<25} {r_def["ruim_aprovado"]:>15,}  {r_opt["ruim_aprovado"]:>15,}')
print(f'  {"Lucro total (BRL)":<25} {r_def["lucro_total"]:>15,.0f}  {r_opt["lucro_total"]:>15,.0f}')
print(f'  {"Ganho ao otimizar":<25}  {r_opt["lucro_total"]-r_def["lucro_total"]:>+14,.0f}')

---
### 5.5 Exemplos de Decisão Individual

In [ ]:
def credit_decision(client_data: dict, model, scaler, features: list,
                    cfg: BusinessConfig) -> dict:
    """
    Recebe dados brutos de um cliente e retorna score + decisão.
    Aplica a mesma feature engineering usada no treino.
    """
    c = dict(client_data)
    # Feature engineering inline (mesma lógica do notebook 02)
    c['loan_to_income']    = c['loan_amount'] / c['income']
    c['monthly_pay_ratio'] = (c['loan_amount']/c['loan_tenure']) / (c['income']/12)
    c['risk_score']        = (1-(c['credit_score']-300)/550)*0.5 + (c['debt_to_income']/2)*0.3
    c['stability_index']   = np.log1p(c['employment_years']) * np.log1p(c['income']/1000)
    c['high_risk_flag']    = int(c['credit_score']<580 and c['debt_to_income']>0.5)
    c['inq_per_account']   = c['num_credit_inq'] / max(c['num_open_accounts'], 1)

    X    = pd.DataFrame([c])[features]
    Xs   = pd.DataFrame(scaler.transform(X), columns=features)
    prob = float(model.predict_proba(Xs)[0, 1])
    dec  = make_decision(prob, cfg)
    band = 'BAIXO' if prob < 0.3 else ('MÉDIO' if prob < 0.6 else 'ALTO')
    return {'prob': prob, 'decisao': dec, 'risco': band}


clientes = [
    {'nome':'Maria (perfil excelente)',
     'age':42,'income':15000,'loan_amount':30000,'loan_tenure':36,
     'credit_score':780,'num_open_accounts':4,'num_credit_inq':0,
     'debt_to_income':0.10,'employment_years':15,'has_mortgage':1,'has_dependents':0},

    {'nome':'Pedro (perfil médio)',
     'age':33,'income':6000,'loan_amount':18000,'loan_tenure':48,
     'credit_score':630,'num_open_accounts':5,'num_credit_inq':3,
     'debt_to_income':0.40,'employment_years':4,'has_mortgage':0,'has_dependents':1},

    {'nome':'João (perfil alto risco)',
     'age':25,'income':2800,'loan_amount':20000,'loan_tenure':60,
     'credit_score':460,'num_open_accounts':9,'num_credit_inq':8,
     'debt_to_income':0.90,'employment_years':0.5,'has_mortgage':0,'has_dependents':1},
]

print('═'*60)
print('  DECISÕES INDIVIDUAIS DE CRÉDITO')
print('═'*60)

for c in clientes:
    nome = c.pop('nome')
    res  = credit_decision(c, best_model, scaler, features, cfg_opt)
    emoji = '✅' if res['decisao']=='APROVAR' else '❌'
    bar   = '█' * int(res['prob']*30)
    print(f'\n  {emoji}  {nome}')
    print(f'     Score   : {res["prob"]:5.1%}  {bar}')
    print(f'     Decisão : {res["decisao"]}')
    print(f'     Risco   : {res["risco"]}')

---
### 5.6 Resumo Final do Pipeline

In [ ]:
# Métricas finais de todos os resultados
all_metrics = {r['name']: r for r in all_results}

print('╔══════════════════════════════════════════════════════════════════╗')
print('║         CREDIT RISK ML — PIPELINE COMPLETO — RESUMO            ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print('║                                                                  ║')
print('║  MODELOS             AUC      F1     Recall  Precision          ║')
for name in names:
    r = all_metrics[name]
    star = '⭐' if name == best_model_name else '  '
    print(f'║  {star} {name:<20} {r["auc"]:.4f}  {r["f1"]:.4f}  {r["recall"]:.4f}  {r["precision"]:.4f}       ║')
print('║                                                                  ║')
print(f'║  MELHOR MODELO      : {best_model_name:<40} ║')
print(f'║  Threshold padrão   : 0.70                                      ║')
print(f'║  Threshold ótimo    : {best_t:.3f} (maximiza lucro)                  ║')
print('║                                                                  ║')
print('║  GARANTIAS DO PIPELINE:                                         ║')
print('║  ✅ Dados sintéticos com padrões realistas de crédito           ║')
print('║  ✅ Limpeza: dedup, imputação por mediana, clipping IQR         ║')
print('║  ✅ 6 features derivadas de conhecimento de domínio             ║')
print('║  ✅ Split estratificado 80/20 — zero data leakage               ║')
print('║  ✅ StandardScaler fit apenas no treino                         ║')
print('║  ✅ class_weight / scale_pos_weight para desbalanceamento       ║')
print('║  ✅ Early stopping no XGBoost                                   ║')
print('║  ✅ Comparação entre 3 modelos com múltiplas métricas           ║')
print('║  ✅ Simulação financeira com threshold configurável             ║')
print('║  ✅ Otimização de threshold por máximo lucro                    ║')
print('║                                                                  ║')
print('║  ARTEFATOS GERADOS:                                             ║')
print('║    models/  → scaler.pkl, *.pkl por modelo, best_model.pkl     ║')
print('║    reports/ → model_comparison.png, feature_importance.png,    ║')
print('║               business_simulation.png, profit_curves.png       ║')
print('║                                                                  ║')
print('║  PRÓXIMOS PASSOS:                                               ║')
print('║    ▸ Optuna para tunagem de hiperparâmetros                     ║')
print('║    ▸ SHAP values para explicabilidade                           ║')
print('║    ▸ Evidently para monitoramento de drift                      ║')
print('║    ▸ Deploy via FastAPI + Docker                                ║')
print('╚══════════════════════════════════════════════════════════════════╝')